# **FLAN-T5 Opinion Summarization**
---

Summarizing multiple opinions/comments using FLAN-T5-large model.

## Loading the Model

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, GenerationConfig

# Load FLAN-T5-large model and tokenizer
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# run in gpu
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="auto"           # Automatically move model to the GPU (CUDA)
)

# run in cpu
# model = AutoModelForSeq2SeqLM.from_pretrained(
#     model_name,
#     dtype=torch.float32  # CPU usage
# )

print(model.config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.57.3",
  "use_cache": true,
  "vocab_size": 32128
}



## Defining the Summarization Function

In [15]:
def create_prompt(comments_text: str, chunks: bool = False) -> str:
    if chunks:
        print("Chunks chunks baby")
        return (
            "Merge these summaries into one cohesive final report.\n"
            f"SUMMARIES:\n{comments_text}\n\n"
            "FINAL COHESIVE SUMMARY:"
        )

    return ( # GOOD PROMPT WITH 0.9 TEMP, OK PROMPT AT 0.8 TEMP
        "Analyze the following social media comments related to climate and environmental issues.\n\n"

        "TASK:\n"
        "1. Identify the overall sentiment polarity (positive, negative, or mixed).\n"
        "2. Describe the dominant emotional tone in one sentence.\n"
        "3. Provide a descriptive summary that preserves the sentiment and emphasizes recurring themes.\n\n"

        "CONSTRAINTS:\n"
        "- Do NOT list individual comments.\n"
        "- Retain the emotional tone expressed by the commenters.\n"
        "- The summary should reflect collective optimism, urgency, or concern if present.\n\n"

        "COMMENTS:\n"
        f"{comments_text}"
    )

    # return (
    #     "Analyze the following climate-related social media comments.\n\n"
    #     "TASK:\n"
    #     "1. Determine overall sentiment (positive, negative, or mixed).\n"
    #     "2. Describe the dominant emotional tone in one sentence.\n"
    #     "3. Write a descriptive summary preserving sentiment and recurring themes.\n\n"
    #     "CONSTRAINTS:\n"
    #     "- Do not list individual comments.\n"
    #     "- Maintain the collective emotional tone.\n\n"
    #     "COMMENTS:\n"
    #     f"{comments_text}"
    # )

    # return ( # Generic summary, a bit biased toward the dominant sentiment, does not capture whole emotional tone
    #     "Below are several social media comments regarding climate change. "
    #     "Summarize the collective voice of these commenters.\n\n"
    #     f"COMMENTS:\n{comments_text}\n\n"
    # )

def chunk_comments(comments, tokenizer, max_tokens=400) -> list[str]:
    """
    Chunk comments into smaller groups based on token count; <400, leaving space for the prompt
    """
    chunks, chunk, tokens = [], [], 0

    for c in comments:
        text = f"- {c}"
        len_t = len(tokenizer.encode(text))

        if chunk and tokens + len_t > max_tokens:
            chunks.append("\n".join(chunk))
            chunk, tokens = [], 0

        chunk.append(text)
        tokens += len_t

    return chunks + (["\n".join(chunk)] if chunk else [])

def generate_summary(prompt, tokenizer, model, gen_config) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    ids = model.generate(
        **inputs,
        generation_config=gen_config
    )

    return tokenizer.decode(ids[0], skip_special_tokens=True)



## Usage
---

**Importing data**

In [3]:
import pandas as pd

df = pd.read_csv("sample_data/cleaned_100.csv")

sample_comments = df["message"]
sample_comments.head()

,message
0,I can't believe people still don't believe in ...
1,Climate change is an interesting hustle; the p...
2,They couldn't predict tomorrow's weather in a ...
3,Global warming is real. Something isn't right ...
4,Not believing in global warming is like opposi...


**Model Configs**


In [13]:
# Creative summary (mostly abstractive)
gen_config = GenerationConfig(
    max_new_tokens=250,        # maximum number of tokens the model can generate
    min_new_tokens=40,         # minimum length to avoid overly short outputs
    do_sample=True,            # enables stochastic (non-greedy) text generation
    temperature=0.9,           # controls randomness; higher = more creative (.85 - 90)
    top_p=0.9,                 # nucleus sampling: keeps tokens within top 90% probability mass
    no_repeat_ngram_size=3,    # prevents repeating any 3-token sequences
    repetition_penalty=1.5     # penalizes repeated tokens to reduce redundancy
)

# Beam search config (leans more on extractive summaries)
# gen_config = GenerationConfig(
#     max_new_tokens=180,
#     min_new_tokens=70,
#     do_sample=False,             # deterministic beam search
#     num_beams=2,                 # higher beam encourages extractive summaries
#     early_stopping=True,         # stop when all beams hit EOS
#     length_penalty=1.1,          # favor concise but complete outputs
#     no_repeat_ngram_size=3,      # avoid repeating phrases
#     repetition_penalty=1.1       # penalize repeated tokens lightly
# )


**Sample Test**

In [5]:
# text = """A fish is an aquatic, anamniotic, gill-bearing vertebrate animal with swimming fins and a hard skull, but lacking limbs with digits. Fish can be grouped into the more basal jawless fish and the more common jawed fish, the latter including all living cartilaginous and bony fish, as well as the extinct placoderms and acanthodians. In a break from the long tradition of grouping all fish into a single class (Pisces), modern phylogenetics views fish as a paraphyletic group which includes all vertebrates except tetrapods. In English, the plural of "fish" is fish when referring to individuals and fishes when referring to species.
# Most fish are cold-blooded, their body temperature varying with the surrounding water, though some large, active swimmers like the white shark and tuna can maintain a higher core temperature. Many fish can communicate acoustically with each other, such as during courtship displays. The study of fish is known as ichthyology."""
# print(generate_summary(text, tokenizer, model, gen_config))

**Summarization procedure (Single Pass or Hierarchical)**


In [16]:
sample_comments = sample_comments[:10]
combined_text = "\n".join(f"- {c}" for c in sample_comments)
token_count = len(tokenizer.encode(combined_text))

# single-pass summarization is possible
if token_count <= 400:
    main_summary = generate_summary(create_prompt(combined_text), tokenizer, model, gen_config)

# Hierarchical summarization
else:
    chunks = chunk_comments(sample_comments, tokenizer)
    chunk_summaries = [
        generate_summary(create_prompt(chunk), tokenizer, model, gen_config)
        for chunk in chunks
    ]
    combined_summaries = "\n".join(f"- {s}" for s in chunk_summaries)
    main_summary = generate_summary(create_prompt(combined_summaries, chunks=True), tokenizer, model, gen_config)

**Output**

In [17]:
print("RAW COMMENTS:")
print(combined_text + "\n")
print(f"Total comments to summarize: {len(sample_comments)}")
print("Token count for input to model:", token_count)

if token_count <= 400:
    print("Summary Type: Single-pass summarization\n")
    print(f"\nSUMMARY:\n{main_summary}")
else:
    print("Summary Type: Hierarchical summarization. Chunking...\n")
    print(f"INTERMEDIATE SUMMARIES:\n{combined_summaries}\n")
    print(f"\nFINAL SUMMARY:\n{main_summary}")

RAW COMMENTS:
- I can't believe people still don't believe in climate change. It's science.
- Climate change is an interesting hustle; the planet stopped warming for 15 years.
- They couldn't predict tomorrow's weather in a 4-month heat wave, yet we trust them on climate change!
- Global warming is real. Something isn't right with the unusually warm weather.
- Not believing in global warming is like opposing peace but expecting the world to be peaceful.
- We are doomed if we do not act. Fixing climate change is our future.
- We just entered an alarming 'new era' of global warming
- global warming real as hell. told us. keep trying to tell us. its 82 degrees and its halloween.
- I swear if it's 80 degrees on Christmas again I will personally defeat global warming. Revenge is a dish best-served cold
- Y'all still don't believe in global warming SMH

Total comments to summarize: 10
Token count for input to model: 221
Summary Type: Single-pass summarization


SUMMARY:
Mixed emotions surrou

## Fine Tuning
---

## Model Evaluation
---